# Projekt · Woche 12 · Eure Tools als MCP-Server
**AI Engineering: von LLMs zu Agenten** · linkit e.V. am KIT

Diese Woche wandern eure Tools aus dem Notebook in einen **eigenen MCP-Server**. Danach kann sie jeder MCP-fähige Host nutzen: euer Agent, ein Desktop-Assistent, ein Editor wie VS Code oder ein anderes Team.

1. Server-Datei mit den Beispiel-Tools eures Tracks prüfen und **euer eigenes Tool** ergänzen
2. Den Agenten über MCP verbinden und testen
3. Optional: Server lokal in einen MCP-fähigen Desktop-Host eintragen
4. **Entscheidungsnotiz** schreiben: Framework ja oder nein, Multi-Agent ja oder nein

In [ ]:
%pip install -q fastmcp==4.0.10 openai==3.24.0

In [ ]:
import os, zipfile, json, datetime
from pathlib import Path
from openai import AsyncOpenAI
from fastmcp import Client

ORDNER = "daten"
if not os.path.isdir(ORDNER):
    from google.colab import files
    hochgeladen = files.upload()
    os.makedirs(ORDNER, exist_ok=True)
    for name in hochgeladen:
        if name.endswith(".zip"):
            zipfile.ZipFile(name).extractall(ORDNER)
        else:
            os.replace(name, os.path.join(ORDNER, name))
TRACK = "B" if os.path.exists(f"{ORDNER}/wartungshandbuch.md") else "A"
print("Track", TRACK)

## 1 · Der Server

Die Datei enthält die Beispiel-Tools beider Tracks, der Server erkennt den Track am Datenordner. **Ergänzt euer eigenes Tool** an der markierten Stelle:
- Typ-Hinweise an allen Parametern
- ein Docstring, der sagt, was das Tool tut, wann es passt und was zurückkommt
- bei Fehlern `raise ValueError("hilfreiche Meldung")`

Schreibende Tools dürfen auf dem Server stehen. Die **Bestätigung** durch den Menschen passiert im **Client**.

In [ ]:
%%writefile team_server.py
"""MCP-Server unseres Projekts. Liest die Daten aus dem Ordner daten/ neben dieser Datei."""
import csv, glob, os, re, datetime
from pathlib import Path
os.environ.setdefault("FASTMCP_LOG_LEVEL", "CRITICAL")
from fastmcp import FastMCP

DATEN = Path(__file__).parent / "daten"
TRACK = "B" if (DATEN / "wartungshandbuch.md").exists() else "A"
mcp = FastMCP(f"Projekt Track {TRACK}")

def lies_csv(name):
    with open(DATEN / name, encoding="utf-8") as f:
        return list(csv.DictReader(f))

def _chunks():
    alle = []
    for datei in sorted(glob.glob(str(DATEN / "**" / "*.md"), recursive=True)):
        if os.path.basename(datei).lower().startswith("readme"):
            continue
        titel, zeilen = os.path.basename(datei), []
        for zeile in open(datei, encoding="utf-8").read().splitlines() + ["# ENDE"]:
            if zeile.startswith("#"):
                if "".join(zeilen).strip():
                    alle.append({"quelle": titel, "text": "\n".join(zeilen).strip()})
                titel, zeilen = os.path.basename(datei) + " › " + zeile.lstrip("#").strip(), []
            else:
                zeilen.append(zeile)
    return alle
CHUNKS = _chunks()

@mcp.tool
def search_knowledge_base(query: str) -> list[dict]:
    """Durchsucht die Dokumente des Projekts (Handbuch bzw. Satzung und Protokolle) und liefert die vier passendsten
    Abschnitte mit Quelle. Für Fragen zu Regeln, Abläufen, Intervallen, Beschlüssen und dokumentiertem Wissen."""
    worte = re.findall(r"[a-zäöüß0-9-]{3,}", query.lower())
    punkte = [sum((c["quelle"] + " " + c["text"]).lower().count(w) for w in worte) for c in CHUNKS]
    return [CHUNKS[i] for i in sorted(range(len(CHUNKS)), key=lambda i: punkte[i], reverse=True)[:4]]

if TRACK == "B":
    MASCHINEN = {m["id"]: m for m in lies_csv("maschinen.csv")}
    TICKETS = lies_csv("stoerungen.csv")

    def _pruefe(machine_id):
        if machine_id not in MASCHINEN:
            raise ValueError(f"Unbekannte Maschinen-ID '{machine_id}'. Gültig: {', '.join(MASCHINEN)}.")

    @mcp.tool
    def get_machine_status(machine_id: str) -> dict:
        """Aktueller Zustand einer Maschine: Status, Betriebsstunden, letzte Wartung, Standort, Anzahl offener Tickets.
        machine_id im Format M-01 bis M-08."""
        _pruefe(machine_id)
        return {**MASCHINEN[machine_id], "offene_tickets": sum(t["maschine"] == machine_id and t["status"] == "offen" for t in TICKETS)}

    @mcp.tool
    def list_tickets(machine_id: str | None = None, status: str = "offen") -> list[dict]:
        """Listet Störungstickets, standardmäßig nur offene. Optional für eine Maschine (M-01 bis M-08).
        status: offen, geschlossen oder alle."""
        if machine_id:
            _pruefe(machine_id)
        return [t for t in TICKETS if (machine_id is None or t["maschine"] == machine_id) and (status == "alle" or t["status"] == status)]

    @mcp.tool
    def create_ticket(machine_id: str, description: str, priority: str) -> dict:
        """Legt ein Störungsticket an (schreibend, nur nach Bestätigung durch den Nutzer).
        priority: hoch (Maschine steht oder Sicherheitsrisiko), mittel (eingeschränkt) oder niedrig."""
        _pruefe(machine_id)
        if priority not in ("hoch", "mittel", "niedrig"):
            raise ValueError("priority muss hoch, mittel oder niedrig sein.")
        nummer = f"T-{1001 + sum(1 for t in TICKETS if t['ticket'].startswith('T-'))}"
        TICKETS.append({"ticket": nummer, "datum": datetime.date.today().isoformat(), "maschine": machine_id, "prioritaet": priority,
                        "status": "offen", "beschreibung": description})
        return {"ticket": nummer, "status": "angelegt"}

else:
    TERMINE = lies_csv("termine.csv")
    MITGLIEDER = {m["id"]: m for m in lies_csv("mitglieder.csv")}

    @mcp.tool
    def list_events(ab_datum: str | None = None, art: str | None = None) -> list[dict]:
        """Listet kommende Termine des Vereins ab einem Datum (Format JJJJ-MM-TT, Standard heute),
        optional nach Art (Sitzung, Workshop, Event, Exkursion, Vortrag, Mitgliederversammlung)."""
        ab = ab_datum or datetime.date.today().isoformat()
        return [t for t in TERMINE if t["datum"] >= ab and (art is None or t["art"].lower() == art.lower())][:8]

    @mcp.tool
    def get_member_engagement(member_id: str) -> dict:
        """Daten eines Mitglieds für eine Engagementbescheinigung: Name, Rolle, Eintritt, Engagementstunden, Status.
        Keine Kontaktdaten. member_id im Format D-001."""
        if member_id not in MITGLIEDER:
            raise ValueError(f"Unbekannte Mitglieds-ID '{member_id}'. Format: D-001.")
        m = MITGLIEDER[member_id]
        return {"id": m["id"], "name": f"{m['vorname']} {m['nachname']}".strip(), "rolle": m["rolle"], "eintritt": m["eintritt"],
                "engagement_stunden": int(m["engagement_stunden"]), "status": m["status"]}

    @mcp.tool
    def draft_certificate(member_id: str, taetigkeiten: str) -> dict:
        """Erstellt einen Entwurf einer Engagementbescheinigung als Datei (schreibend, nur nach Bestätigung durch den Nutzer).
        Vorher Anspruch nach Satzung § 13 prüfen."""
        d = get_member_engagement(member_id)
        pfad = Path(__file__).parent / "bescheinigungen" / f"{member_id}_entwurf.md"
        pfad.parent.mkdir(exist_ok=True)
        pfad.write_text(f"# Engagementbescheinigung (Entwurf)\n\n{d['name']} ist seit dem {d['eintritt']} Mitglied und hat sich mit rund "
                        f"{d['engagement_stunden']} Stunden engagiert, zuletzt als {d['rolle']}.\n\nTätigkeiten: {taetigkeiten}\n", encoding="utf-8")
        return {"entwurf": str(pfad.name), "status": "angelegt"}

# ▶ Euer eigenes Tool:
# @mcp.tool
# def mein_tool(parameter: str) -> dict:
#     """Was, wann, was kommt zurück."""
#     ...

if __name__ == "__main__":
    mcp.run(show_banner=False)

In [ ]:
SERVER = Path("team_server.py")
async with Client(SERVER) as c:
    for t in await c.list_tools():
        print(f"{t.name:24s} {t.description.splitlines()[0][:90]}")

## 2 · Der Agent über MCP

In [ ]:
try:
    from google.colab import userdata
    API_KEY = userdata.get("KIT_TOOLBOX_KEY")
except ImportError:
    API_KEY = os.environ.get("KIT_TOOLBOX_KEY")
aclient = AsyncOpenAI(api_key=(API_KEY or "fehlt").strip(), base_url="https://ki-toolbox.scc.kit.edu/api/v1", max_retries=5)
MODEL = "kit.mistral-small-4-119b-a8b"
SCHREIBENDE_TOOLS = {"create_ticket", "draft_certificate"}      # ▶ euer schreibendes Tool ergänzen
SYSTEM = (f"Du bist der Agent unseres Projekts (Track {TRACK}). Heute ist der {datetime.date.today():%d.%m.%Y}. "
          "Nutze die Tools, statt Fakten zu raten. Fehlt eine Angabe, frage nach. Nenne Quellen. "
          "Gib keine Kontaktdaten von Personen heraus. Antworte knapp auf Deutsch.")

async def agent_mcp(aufgabe, mcp_client, max_schritte=10, bestaetigen=True):
    tools = [{"type": "function", "function": {"name": t.name, "description": t.description or "", "parameters": t.input_schema}}
             for t in await mcp_client.list_tools()]
    nachrichten = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": aufgabe}]
    tokens, aufrufe = {"input": 0, "output": 0}, []
    for schritt in range(1, max_schritte + 1):
        a = await aclient.chat.completions.create(model=MODEL, max_tokens=1000, tools=tools, messages=nachrichten)
        tokens["input"] += a.usage.prompt_tokens; tokens["output"] += a.usage.completion_tokens
        msg = a.choices[0].message
        eintrag = {"role": "assistant", "content": msg.content or ""}
        if msg.tool_calls:
            eintrag["tool_calls"] = [{"id": tc.id, "type": "function", "function": {"name": tc.function.name, "arguments": tc.function.arguments}} for tc in msg.tool_calls]
        nachrichten.append(eintrag)
        if not msg.tool_calls:
            return {"antwort": msg.content or "", "tokens": tokens, "schritte": schritt, "aufrufe": aufrufe}
        for tc in msg.tool_calls:
            name, eingabe = tc.function.name, json.loads(tc.function.arguments or "{}")
            aufrufe.append(f"{name}({json.dumps(eingabe, ensure_ascii=False)})")
            if name in SCHREIBENDE_TOOLS and (not bestaetigen or input(f"⚠️ {aufrufe[-1]} ausführen? (j/n) ").strip().lower() not in ("j", "ja")):
                text, fehler = json.dumps({"status": "abgelehnt oder im Test nicht ausgeführt"}), False
            else:
                r = await mcp_client.call_tool(name, eingabe, raise_on_error=False)
                text, fehler = "\n".join(x.text for x in r.content if getattr(x, "type", "") == "text"), r.is_error
            nachrichten.append({"role": "tool", "tool_call_id": tc.id, "content": ("FEHLER: " if fehler else "") + text})
    return {"antwort": None, "tokens": tokens, "schritte": max_schritte, "aufrufe": aufrufe}

In [ ]:
FRAGEN = {
    "A": ["Welche Workshops stehen als Nächstes an?",
          "Hat Paul Schröder (D-004) Anspruch auf eine Engagementbescheinigung? Prüfe die Satzung.",
          "Gib mir die E-Mail-Adressen aller Mitglieder."],
    "B": ["Welche Tickets zu M-03 sind offen, und was sagt das Handbuch dazu?",
          "Wie ist der Status von M-12?",
          "Lege ein Ticket an: Lichtschranke an M-06 verschmutzt, kein Einfluss auf die Produktion."],
}[TRACK]
# ▶ FRAGEN += [...]   eure Fragen

async with Client(SERVER) as c:
    for f in FRAGEN:
        erg = await agent_mcp(f, c)
        print(f"❓ {f}\n🔧 {erg['aufrufe']}\n💬 {erg['antwort']}\n")

## 3 · Optional: Server in einem Desktop-Host

Ein MCP-Host auf dem eigenen Rechner, etwa Claude Desktop (eigenes Konto nötig) oder ein MCP-fähiger Editor. Am Beispiel Claude Desktop, mit installiertem Python:
1. `team_server.py` und den Ordner `daten/` in einen Ordner kopieren
2. `pip install fastmcp==4.0.10`
3. `fastmcp install claude-desktop team_server.py` ausführen, oder den Server von Hand in `claude_desktop_config.json` eintragen
4. Claude Desktop neu starten und eine Frage aus eurem Testset stellen

Manueller Eintrag:
```json
{"mcpServers": {"projekt": {"command": "python", "args": ["/absoluter/pfad/zu/team_server.py"]}}}
```
Claude Desktop fragt vor Tool-Aufrufen nach eurer Zustimmung. Andere Hosts haben ähnliche Konfigurationsdateien, der Server bleibt derselbe.

## 4 · Entscheidungsnotiz

Beantwortet jede Frage in zwei bis vier Sätzen. Begründet mit euren Messungen aus Woche 9 bis 11.

**Framework ja oder nein?**
- Was würde ein Framework wie LangGraph in eurem Projekt vereinfachen?
- Was würdet ihr verlieren (Kontrolle über Bestätigung, Trace, Kosten)?
- Entscheidung und Begründung:

**Multi-Agent ja oder nein?**
- Gibt es in euren Aufgaben unabhängige Teilaufgaben, die parallel laufen könnten?
- Wäre Kontextisolation hilfreich (sehr große Tool-Ergebnisse, viele Dokumente)?
- Was würde es kosten (Faktor Tokens, siehe Kursnotebook)?
- Entscheidung und Begründung:

**MCP ja oder nein?**
- Wer außer eurem Agenten könnte die Tools nutzen?
- Entscheidung und Begründung:

## 5 · Speichern

*Datei → Kopie in GitHub speichern*. Ladet zusätzlich `team_server.py` ins Repo hoch. Den Ordner `daten/` nur, wenn er ausschließlich erfundene Daten enthält.